# LLT LoRA sweep, part a2

Run Library runs for the lltLoRA lab. Writes res_*.json and svd_*.json.

In [ ]:

import json, os, subprocess, sys, time, platform
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers==5.22.1", "tokenizers==0.23.3",
                "huggingface_hub==2.4.0", "peft==0.21.2", "accelerate==1.15.0", "bitsandbytes==0.50.2",
                "datasets==5.1.0"], check=False)
open("/kaggle/working/lora_lab.py", "w").write('# lora_lab.py - data, training loop and evaluation for the lltLoRA lab.\n# The notebook writes this file; our recorded runs (kaggle/lora_worker.py) import the same file.\nimport math\nimport re\nimport time\n\nimport torch\nfrom datasets import load_dataset\n\nMODEL = "Qwen/Qwen3.5-0.8B-Base"\nPROMPT = """Question: {q}\nAnswer:"""\n\n\ndef load_gsm8k(split, n=None):\n    rows = load_dataset("openai/gsm8k", "main", split=split)\n    if n is not None:\n        rows = rows.select(range(n))\n    # Drop the calculator notes such as <<48/2=24>> from each worked answer.\n    return [(PROMPT.format(q=r["question"]), " " + re.sub("<<[^>]*>>", "", r["answer"])) for r in rows]\n\n\ndef encode(tok, pairs):\n    rows = []\n    for prompt, answer in pairs:\n        p = tok(prompt)["input_ids"]\n        a = tok(answer)["input_ids"] + [tok.eos_token_id]\n        rows.append((p + a, [-100] * len(p) + a))      # loss only on the answer and its end token\n    return rows\n\n\ndef text_rows(tok, n_rows=32, T=512):\n    # General text to measure forgetting: Wikipedia articles the task never shows.\n    ds = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split="test")\n    ids = tok("".join(ds["text"]))["input_ids"][: n_rows * T]\n    return [(ids[i:i + T], ids[i:i + T]) for i in range(0, n_rows * T, T)]\n\n\ndef micro_batches(rows, budget):\n    # Group rows so that rows x longest row stays within budget tokens. Memory follows that product,\n    # because the logits tensor is rows x length x 248,320 vocabulary entries.\n    rows = sorted(rows, key=lambda r: len(r[0]))\n    groups, cur = [], []\n    for r in rows:\n        if cur and (len(cur) + 1) * len(r[0]) > budget:\n            groups.append(cur)\n            cur = []\n        cur.append(r)\n    return groups + [cur]\n\n\ndef collate(rows, pad_id, device):\n    T = max(len(ids) for ids, _ in rows)\n    ids = torch.full((len(rows), T), pad_id, dtype=torch.long)\n    labels = torch.full((len(rows), T), -100, dtype=torch.long)\n    mask = torch.zeros((len(rows), T), dtype=torch.long)\n    for i, (x, y) in enumerate(rows):\n        ids[i, :len(x)] = torch.tensor(x)\n        labels[i, :len(y)] = torch.tensor(y)\n        mask[i, :len(x)] = 1\n    return ids.to(device), mask.to(device), labels.to(device)\n\n\ndef amp(device):\n    return torch.autocast(device, dtype=torch.float16, enabled=device == "cuda")\n\n\n@torch.no_grad()\ndef mean_loss(model, rows, pad_id, device, budget=2048):\n    # Mean next-token loss over every label that is not -100, in nats per token.\n    was_training = model.training\n    model.eval()\n    total, n = 0.0, 0\n    for group in micro_batches(rows, budget):\n        ids, mask, labels = collate(group, pad_id, device)\n        k = int((labels[:, 1:] != -100).sum())\n        with amp(device):\n            loss = model(input_ids=ids, attention_mask=mask, labels=labels, num_items_in_batch=k).loss\n        total += loss.item() * k\n        n += k\n    model.train(was_training)\n    return total / n\n\n\ndef train(model, rows, steps, opt, pad_id, device="cuda", batch=16, budget=1024, warmup=10, seed=0,\n          on_step=None):\n    params = [p for p in model.parameters() if p.requires_grad]\n    # Linear warmup over `warmup` steps, then linear decay to 0 at the last step.\n    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warmup) * (1 - s / steps))\n    # Qwen3.5\'s fp16 gradients overflow at the default scale of 65,536, so start lower.\n    scaler = torch.amp.GradScaler(device, init_scale=2048, enabled=device == "cuda")\n    g = torch.Generator().manual_seed(seed)\n    order, history = [], []\n    model.train()\n    for step in range(steps):\n        t0 = time.time()\n        if len(order) < batch:\n            order += torch.randperm(len(rows), generator=g).tolist()\n        picked = [rows[i] for i in order[:batch]]\n        order = order[batch:]\n        n_tok = sum(sum(1 for t in y if t != -100) for _, y in picked)\n        loss_sum = 0.0\n        for group in micro_batches(picked, budget):\n            ids, mask, labels = collate(group, pad_id, device)\n            with amp(device):\n                # num_items_in_batch makes the loss a sum over this micro-batch divided by the\n                # answer tokens of the whole batch, so every token weighs the same.\n                loss = model(input_ids=ids, attention_mask=mask, labels=labels, num_items_in_batch=n_tok).loss\n            scaler.scale(loss).backward()\n            loss_sum += loss.item()\n        scale = scaler.get_scale()\n        scaler.unscale_(opt)\n        gnorm = float(torch.nn.utils.clip_grad_norm_(params, 1.0))\n        scaler.step(opt)\n        scaler.update()\n        opt.zero_grad(set_to_none=True)\n        sched.step()\n        if device == "cuda":\n            torch.cuda.synchronize()\n        rec = {"step": step + 1, "trainLoss": round(loss_sum, 5), "gradNorm": round(gnorm, 4),\n               "lr": sched.get_last_lr()[0], "skipped": scaler.get_scale() < scale,\n               "sec": round(time.time() - t0, 3), "tokens": n_tok}\n        history.append(rec)\n        if on_step:\n            on_step(rec)\n    return history\n\n\nNUMBER = "-?[0-9][0-9,]*(?:[.][0-9]+)?"\n\n\ndef final_number(text, strict=True):\n    # strict: the number after ####, the answer format of the training data.\n    # lenient: that number, or else the last number before the model starts a new question.\n    text = text.split("Question:")[0]\n    m = re.search("####[ ]*(" + NUMBER + ")", text)\n    if m:\n        return m.group(1).replace(",", "").rstrip(".")\n    if strict:\n        return None\n    found = re.findall(NUMBER, text)\n    return found[-1].replace(",", "").rstrip(".") if found else None\n\n\n@torch.no_grad()\ndef accuracy(model, tok, pairs, device="cuda", batch=50, max_new=256):\n    # Greedy answers; a problem counts as solved when the number after #### matches the reference.\n    was_training = model.training\n    model.eval()\n    tok.padding_side = "left"\n    strict, lenient, outputs = 0, 0, []\n    for i in range(0, len(pairs), batch):\n        chunk = pairs[i:i + batch]\n        enc = tok([p for p, _ in chunk], return_tensors="pt", padding=True).to(device)\n        with amp(device):\n            out = model.generate(**enc, max_new_tokens=max_new, do_sample=False, use_cache=True,\n                                 pad_token_id=tok.pad_token_id)\n        for (_, ref), seq in zip(chunk, out[:, enc["input_ids"].shape[1]:]):\n            text = tok.decode(seq, skip_special_tokens=True)\n            gold = final_number(ref)\n            strict += final_number(text) == gold\n            lenient += final_number(text, strict=False) == gold\n            outputs.append(text)\n    tok.padding_side = "right"\n    model.train(was_training)\n    return strict / len(pairs), lenient / len(pairs), outputs\n')
open("/kaggle/working/lora_worker.py", "w").write('"""One lltLoRA run in its own process. Usage: python lora_worker.py <job.json> <result.json>\n\njob keys: tag, model, mode (full|lora|eval), lr, opt (adamw8bit|adamw32), r, alpha, targets (all|attn|mlp),\nrslora, b_init (zero|random), n_train, steps (default n_train // batch), batch, budget, seed, pad_to,\neval_every, n_heldout, n_wiki, n_acc, spectra, truncate (list of ranks), save_adapter, pool (list of module\nnames whose pooled singular vectors are saved for the overview).\n"""\nimport gc, json, os, sys, time, traceback, platform, io, logging\nos.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")\nos.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")\njob = json.load(open(sys.argv[1])); outp = sys.argv[2]\nR = {"tag": job["tag"], "job": job}\ndef save():\n    json.dump(R, open(outp, "w"), indent=1)\nimport torch, transformers, peft, datasets\nsys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))\nimport lora_lab as L\nCUDA = os.environ.get("LORA_DEV", "cuda") == "cuda"\ndef gb(f):\n    return round(f() / 1e9, 2) if CUDA else 0.0\nfrom transformers import AutoModelForCausalLM, AutoTokenizer\nlogbuf = io.StringIO(); h = logging.StreamHandler(logbuf); h.setLevel(logging.WARNING)\nlogging.getLogger("transformers").addHandler(h)\n\nATTN = ["q_proj", "k_proj", "v_proj", "o_proj", "in_proj_qkv", "in_proj_z", "in_proj_a", "in_proj_b", "out_proj"]\nMLP = ["gate_proj", "up_proj", "down_proj"]\n\n\ndef linear_names(model):\n    return [n for n, m in model.named_modules() if isinstance(m, torch.nn.Linear) and not n.endswith("lm_head")]\n\n\ndef evals(model, tok, held, wiki, acc_pairs, dev, tag=""):\n    t = time.time()\n    e = {"heldoutLoss": round(L.mean_loss(model, held, tok.pad_token_id, dev), 5),\n         "wikiLoss": round(L.mean_loss(model, wiki, tok.pad_token_id, dev), 5)}\n    if acc_pairs:\n        s, l, outs = L.accuracy(model, tok, acc_pairs, dev)\n        e.update(accStrict=round(s, 4), accLenient=round(l, 4), samples=outs[:3])\n    e["evalSec"] = round(time.time() - t, 1)\n    print("eval", tag, {k: v for k, v in e.items() if k != "samples"}, flush=True)\n    return e\n\n\ntry:\n    dev = os.environ.get("LORA_DEV", "cuda")\n    R["env"] = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,\n                "transformers": transformers.__version__, "peft": peft.__version__, "datasets": datasets.__version__,\n                "gpu": torch.cuda.get_device_name(0) if dev == "cuda" else "cpu"}\n    try:\n        import bitsandbytes; R["env"]["bitsandbytes"] = bitsandbytes.__version__\n    except Exception:\n        pass\n    batch = job.get("batch", 16); n_train = job.get("n_train", 2000)\n    steps = job.get("steps") or n_train // batch\n    tok = AutoTokenizer.from_pretrained(job["model"])\n    pairs = L.load_gsm8k("train")\n    import random\n    random.Random(1234).shuffle(pairs)                     # one fixed shuffle; every run draws its subset from it\n    rows = L.encode(tok, pairs[:n_train])\n    if job.get("pad_to"):\n        P = job["pad_to"]\n        rows = [(x + [tok.pad_token_id] * (P - len(x)), y + [-100] * (P - len(y))) for x, y in rows]\n    test = L.load_gsm8k("test")\n    held = L.encode(tok, test[:job.get("n_heldout", 300)])\n    held_small = held[:100]\n    wiki = L.text_rows(tok, job.get("n_wiki", 32))\n    acc_pairs = test[:job.get("n_acc", 200)] if job.get("n_acc", 200) else []\n    R["data"] = {"nTrain": len(rows), "trainAnswerTokens": sum(sum(1 for t in y if t != -100) for _, y in rows),\n                 "trainTokens": sum(len(x) for x, _ in rows), "heldout": len(held), "wikiRows": len(wiki),\n                 "accProblems": len(acc_pairs)}\n    t0 = time.time()\n    model = AutoModelForCausalLM.from_pretrained(job["model"], dtype=torch.float32, device_map=dev)\n    R["loadSec"] = round(time.time() - t0, 1)\n    R["totalParamsBase"] = sum(p.numel() for p in model.parameters())\n    model.config.use_cache = False\n    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n    lin = linear_names(model)\n    W0 = None\n    if job["mode"] == "full" and (job.get("spectra") or job.get("truncate")):\n        W0 = {n: p.detach().to("cpu", copy=True) for n, p in model.named_parameters()}\n    if job.get("eval_base"):\n        R["evalBase"] = evals(model, tok, held, wiki, acc_pairs, dev, "base")\n    if job["mode"] == "lora":\n        from peft import LoraConfig, get_peft_model\n        tgt = {"all": "all-linear", "attn": ATTN, "mlp": MLP}[job.get("targets", "all")]\n        cfg = LoraConfig(r=job["r"], lora_alpha=job.get("alpha", 32), lora_dropout=job.get("dropout", 0.0),\n                         target_modules=tgt, use_rslora=job.get("rslora", False),\n                         init_lora_weights=job.get("b_init", "zero") == "zero", task_type="CAUSAL_LM")\n        model.enable_input_require_grads()\n        torch.manual_seed(job.get("seed", 0))\n        model = get_peft_model(model, cfg)\n        for p in model.parameters():\n            if p.requires_grad and p.dtype != torch.float32:\n                p.data = p.data.float()\n    params = [p for p in model.parameters() if p.requires_grad]\n    R["trainableParams"] = sum(p.numel() for p in params)\n    R["totalParams"] = sum(p.numel() for p in model.parameters())\n    if job["mode"] != "eval":\n        if job.get("opt", "adamw32") == "adamw8bit":\n            import bitsandbytes as bnb\n            opt = bnb.optim.AdamW8bit(params, lr=job["lr"], weight_decay=0.0)\n        elif job.get("opt") == "adamw32-loop":\n            # one tensor at a time: no multi-tensor temporaries, the same maths as the default AdamW\n            opt = torch.optim.AdamW(params, lr=job["lr"], weight_decay=0.0, foreach=False)\n        else:\n            opt = torch.optim.AdamW(params, lr=job["lr"], weight_decay=0.0)\n    R["memAfterLoadGB"] = gb(torch.cuda.memory_allocated)\n    curve = []\n    if job.get("eval_every"):\n        curve.append({"step": 0, "heldoutLoss": round(L.mean_loss(model, held_small, tok.pad_token_id, dev), 5)})\n        print("step 0", curve[-1], flush=True)\n    save()\n    hist = []\n    if job["mode"] != "eval":\n        if CUDA:\n            torch.cuda.reset_peak_memory_stats()\n        ee = job.get("eval_every") or 0\n        def on_step(rec):\n            rec["memGB"] = gb(torch.cuda.max_memory_allocated)\n            if rec["step"] % 10 == 0 or rec["step"] <= 3:\n                print(rec, flush=True)\n            if ee and rec["step"] % ee == 0 and rec["step"] < steps:\n                curve.append({"step": rec["step"], "heldoutLoss": round(L.mean_loss(model, held_small, tok.pad_token_id, dev), 5)})\n                print("curve", curve[-1], flush=True)\n        t1 = time.time()\n        hist = L.train(model, rows, steps, opt, tok.pad_token_id, dev, batch=batch, budget=job.get("budget", 1024),\n                       warmup=job.get("warmup", 10), seed=job.get("seed", 0), on_step=on_step)\n        R["trainSec"] = round(time.time() - t1, 1)\n        R["peakMemGB"] = gb(torch.cuda.max_memory_allocated)\n        R["peakReservedGB"] = gb(torch.cuda.max_memory_reserved)\n        secs = [h["sec"] for h in hist[3:]] or [h["sec"] for h in hist]\n        R["secPerStepMedian"] = round(sorted(secs)[len(secs) // 2], 3)\n        R["skippedSteps"] = sum(h["skipped"] for h in hist)\n        R["nonFinite"] = sum(1 for h in hist if not (h["trainLoss"] == h["trainLoss"]) or abs(h["trainLoss"]) == float("inf"))\n    R["metrics"] = hist\n    R["curve"] = curve\n    save()\n    if job.get("eval_final", True):\n        R["evalFinal"] = evals(model, tok, held, wiki, acc_pairs, dev, "final")\n        if job.get("eval_every"):\n            # same 100 examples as the earlier curve points; evalFinal uses all n_heldout\n            curve.append({"step": steps, "heldoutLoss": round(L.mean_loss(model, held_small, tok.pad_token_id, dev), 5)})\n    save()\n    if job.get("save_adapter") and job["mode"] == "lora":\n        model.save_pretrained(job["save_adapter"])\n        R["adapterBytes"] = sum(os.path.getsize(os.path.join(job["save_adapter"], f)) for f in os.listdir(job["save_adapter"]))\n    if job.get("dropout_eval_check"):\n        # LoRA dropout left on at evaluation: score the same held-out set twice in train mode.\n        model.train()\n        vals = []\n        for _ in range(2):\n            tot, n = 0.0, 0\n            with torch.no_grad():\n                for group in L.micro_batches(held_small, 2048):\n                    ids, mask, labels = L.collate(group, tok.pad_token_id, dev)\n                    k = int((labels[:, 1:] != -100).sum())\n                    with L.amp(dev):\n                        tot += model(input_ids=ids, attention_mask=mask, labels=labels, num_items_in_batch=k).loss.item() * k\n                    n += k\n            vals.append(round(tot / n, 5))\n        R["dropoutEvalTrainMode"] = vals\n        R["dropoutEvalEvalMode"] = round(L.mean_loss(model, held_small, tok.pad_token_id, dev), 5)\n        save()\n    if job["mode"] == "full" and W0 is not None:\n        sd = dict(model.named_parameters())\n        spec = {}; pooled = {}\n        lin_set = set(lin)\n        tops = {}\n        for n in lin:\n            W = sd[n + ".weight"].detach()\n            d = W.float() - W0[n + ".weight"].to(dev)\n            fro = float(d.norm())\n            if job.get("truncate"):\n                U, S, Vh = torch.linalg.svd(d, full_matrices=False)\n                K = min(256, S.numel())\n                tops[n] = (U[:, :K].cpu(), S.cpu(), Vh[:K].cpu())\n            else:\n                S = torch.linalg.svdvals(d)\n            s = S.cpu()\n            e = (s ** 2).cumsum(0) / (s ** 2).sum()\n            spec[n] = {"shape": list(d.shape), "fro": fro, "relChange": fro / float(W0[n + ".weight"].norm()),\n                       "s": [float(f"{v:.5g}") for v in s.tolist()],\n                       "energy": {str(r): round(float(e[min(r, len(e)) - 1]), 5) for r in (1, 2, 4, 8, 16, 32, 64, 128, 256)},\n                       "r90": int((e < 0.9).sum()) + 1, "r50": int((e < 0.5).sum()) + 1}\n            if n in job.get("pool", []):\n                # Pool rows and columns into 64 blocks each. Pooling is linear, so the pooled rank-r\n                # reconstruction equals the sum over the top r of (pooled u_i) s_i (pooled v_i).\n                Uf, Sf, Vhf = torch.linalg.svd(d, full_matrices=False)\n                m, k = d.shape\n                pu = Uf[:, :64].reshape(64, m // 64, 64).mean(1) if m % 64 == 0 else None\n                pv = Vhf[:64].T.reshape(64, k // 64, 64).mean(1) if k % 64 == 0 else None\n                pooled[n] = {"shape": [m, k], "s": [float(f"{v:.6g}") for v in Sf[:64].tolist()],\n                             "U64": [[round(float(v), 6) for v in row] for row in pu.tolist()] if pu is not None else None,\n                             "V64": [[round(float(v), 6) for v in row] for row in pv.tolist()] if pv is not None else None,\n                             "pooledFull": [[round(float(v), 7) for v in row] for row in\n                                            d.reshape(64, m // 64, 64, k // 64).mean((1, 3)).tolist()]}\n            del d\n        other = {}\n        for n, p in sd.items():\n            if n.endswith(".weight") and n[:-7] in lin_set:\n                continue\n            dd = (p.detach().float() - W0[n].to(dev))\n            other[n] = {"shape": list(p.shape), "fro": float(dd.norm()), "relChange": float(dd.norm()) / max(float(W0[n].norm()), 1e-12)}\n        json.dump({"tag": job["tag"], "linear": spec, "other": other, "pooled": pooled},\n                  open(outp.replace("res_", "svd_"), "w"))\n        R["spectraFile"] = os.path.basename(outp.replace("res_", "svd_"))\n        save()\n        if job.get("truncate"):\n            # Rebuild the model from the base plus rank-r pieces of each linear delta. Embeddings and norms go\n            # back to the base, because all-linear LoRA cannot change them.\n            ft = {n: p.detach().to("cpu", copy=True) for n, p in sd.items()}\n            res = []\n            for r in job["truncate"] + ["full"]:\n                with torch.no_grad():\n                    for n, p in sd.items():\n                        if n.endswith(".weight") and n[:-7] in lin_set:\n                            U, S, Vh = tops[n[:-7]]\n                            if r == "full":\n                                p.copy_(ft[n].to(dev))\n                            else:\n                                k = min(r, U.shape[1])\n                                p.copy_(W0[n].to(dev) + (U[:, :k].to(dev) * S[:k].to(dev)) @ Vh[:k].to(dev))\n                        else:\n                            p.copy_(W0[n].to(dev))\n                e = evals(model, tok, held, wiki, acc_pairs if job.get("truncate_acc") else [], dev, f"trunc r={r}")\n                e["rank"] = r\n                res.append(e); R["truncation"] = res; save()\n    R["warnings"] = logbuf.getvalue()[-2000:]\n    R["ok"] = True\nexcept Exception as e:\n    R["ok"] = False; R["error"] = f"{type(e).__name__}: {str(e)[:900]}"; R["trace"] = traceback.format_exc()[-2500:]\n    try:\n        R["peakMemGB"] = gb(torch.cuda.max_memory_allocated)\n    except Exception:\n        pass\n    R["warnings"] = logbuf.getvalue()[-2000:]\nsave()\nprint("RESULT", {k: R.get(k) for k in ("tag", "ok", "peakMemGB", "secPerStepMedian", "trainableParams", "error")}, flush=True)\n')
import torch, transformers, peft, bitsandbytes, datasets
ENV = {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda,
       "transformers": transformers.__version__, "peft": peft.__version__, "bitsandbytes": bitsandbytes.__version__,
       "datasets": datasets.__version__, "gpu": torch.cuda.get_device_name(0), "gpuCount": torch.cuda.device_count()}
print(ENV, flush=True)
T0 = time.time()
DONE = []
def job(**kw):
    tag = kw["tag"]
    jf = f"/kaggle/working/job_{tag}.json"; rf = f"/kaggle/working/res_{tag}.json"
    json.dump(kw, open(jf, "w"))
    t = time.time()
    p = subprocess.run([sys.executable, "/kaggle/working/lora_worker.py", jf, rf], capture_output=True, text=True)
    open(f"/kaggle/working/log_{tag}.txt", "w").write((p.stdout or "") + "\n---stderr---\n" + (p.stderr or "")[-6000:])
    try:
        r = json.load(open(rf))
    except Exception:
        r = {"tag": tag, "ok": False, "error": "no result file, worker crashed", "stderr": (p.stderr or "")[-3000:]}
        json.dump(r, open(rf, "w"))
    DONE.append(tag)
    print(tag, {k: r.get(k) for k in ("ok", "peakMemGB", "secPerStepMedian", "trainSec", "skippedSteps", "error")},
          "| final", {k: v for k, v in (r.get("evalFinal") or {}).items() if k != "samples"},
          "| wall min", round((time.time() - t) / 60, 1), "| total min", round((time.time() - T0) / 60, 1), flush=True)
    json.dump({"env": ENV, "done": DONE, "minutes": round((time.time() - T0) / 60, 1)}, open("/kaggle/working/summary.json", "w"))
    return r


In [ ]:

Q35 = "Qwen/Qwen3.5-0.8B-Base"
POOL = ["model.layers.11.self_attn.v_proj", "model.layers.11.mlp.down_proj", "model.layers.11.self_attn.q_proj"]
C = dict(model=Q35, n_train=2000, batch=16, eval_every=25, n_heldout=300, n_wiki=32, n_acc=200, warmup=10)
FULL = dict(C, mode="full", opt="adamw8bit", budget=1024, spectra=True, pool=POOL)
LORA = dict(C, mode="lora", r=16, alpha=32, targets="all", budget=2048)


In [ ]:

# Part A2 + B in one session: the LoRA learning-rate sweep, then every run that needs its best rate.
lr_res = {}
for lr in [3e-5, 1e-4, 3e-4, 1e-3, 3e-3]:
    r = job(tag=f"lora-r16-lr{lr:g}-s0", lr=lr, seed=0, **LORA)
    if r.get("ok"):
        lr_res[lr] = r["evalFinal"]["heldoutLoss"]
L = min(lr_res, key=lr_res.get)
print("BEST LORA LR", L, lr_res, flush=True)
json.dump({"bestLoraLr": L, "heldout": {str(k): v for k, v in lr_res.items()}}, open("/kaggle/working/best_lr.json", "w"))
for r_ in [1, 4, 64, 256]:
    job(tag=f"lora-r{r_}-s0", lr=L, seed=0, **dict(LORA, r=r_))
job(tag="lora-attn-r16-s0", lr=L, seed=0, **dict(LORA, targets="attn"))
job(tag="lora-mlp-r16-s0", lr=L, seed=0, **dict(LORA, targets="mlp"))
job(tag="lora-r256-rslora-s0", lr=L, seed=0, **dict(LORA, r=256, rslora=True))
job(tag="lora-r16-s1", lr=L, seed=1, **dict(LORA, save_adapter="/kaggle/working/adapter-r16-s1"))
job(tag="lora-attn-r16-s1", lr=L, seed=1, **dict(LORA, targets="attn"))
job(tag="lora-r1-s1", lr=L, seed=1, **dict(LORA, r=1))
job(tag="lora-r16-n500-s0", lr=L, seed=0, **dict(LORA, n_train=500))
job(tag="broken-binit-random", lr=L, seed=0, **dict(LORA, b_init="random", steps=30, eval_every=10, n_acc=0))
job(tag="broken-dropout-at-eval", lr=L, seed=0, **dict(LORA, dropout=0.1, steps=30, n_acc=0, dropout_eval_check=True))
job(tag="full32loop-lr1e-05-s0", lr=1e-5, seed=0, **dict(FULL, opt="adamw32-loop", budget=512, spectra=False))

print("TOTAL MIN", round((time.time() - T0) / 60, 1))
